In [0]:
%sql
-- SELECT * FROM anvisa.b_dados_abertos_medicamentos LIMIT 10;
-- SELECT * FROM anvisa.b_vigimed_medicamentos LIMIT 10;
-- SELECT * FROM anvisa.b_vigimed_reacoes LIMIT 10;
-- SELECT * FROM anvisa.b_vigimed_notificacoes LIMIT 10;
--

In [0]:
%sql
CREATE OR REPLACE TABLE anvisa.s_dados_abertos_medicamentos AS

WITH transformed_data AS (
  SELECT
    -- 1. Colunas de Texto: Trim, Maiúsculas e Remoção de Acentos via translate()
    translate(
      upper(trim(TIPO_PRODUTO)),
      'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ',
      'AAAAAEEEEIIIIOOOOOUUUUCN'
    ) AS TIPO_PRODUTO,

    translate(
      upper(trim(NOME_PRODUTO)),
      'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ',
      'AAAAAEEEEIIIIOOOOOUUUUCN'
    ) AS NOME_PRODUTO,

    translate(
      upper(trim(CATEGORIA_REGULATORIA)),
      'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ',
      'AAAAAEEEEIIIIOOOOOUUUUCN'
    ) AS CATEGORIA_REGULATORIA,

    translate(
      upper(trim(CLASSE_TERAPEUTICA)),
      'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ',
      'AAAAAEEEEIIIIOOOOOUUUUCN'
    ) AS CLASSE_TERAPEUTICA,

    translate(
      upper(trim(EMPRESA_DETENTORA_REGISTRO)),
      'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ',
      'AAAAAEEEEIIIIOOOOOUUUUCN'
    ) AS EMPRESA_DETENTORA_REGISTRO,

    translate(
      upper(trim(SITUACAO_REGISTRO)),
      'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ',
      'AAAAAEEEEIIIIOOOOOUUUUCN'
    ) AS SITUACAO_REGISTRO,

    translate(
      upper(trim(PRINCIPIO_ATIVO)),
      'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ',
      'AAAAAEEEEIIIIOOOOOUUUUCN'
    ) AS PRINCIPIO_ATIVO,

    -- 2. Chaves e Identificadores: Apenas dígitos, mantidos como STRING
    regexp_replace(trim(NUMERO_REGISTRO_PRODUTO), '[^0-9]', '') AS NUMERO_REGISTRO_PRODUTO,
    regexp_replace(trim(NUMERO_PROCESSO), '[^0-9]', '')          AS NUMERO_PROCESSO,

    -- 3. Datas: Conversão segura (coalesce com múltiplos formatos e suporte a MMyyyy)
    coalesce(
      try_to_date(DATA_FINALIZACAO_PROCESSO, 'dd/MM/yyyy'),
      try_to_date(DATA_FINALIZACAO_PROCESSO, 'yyyy-MM-dd'),
      try_to_date(DATA_FINALIZACAO_PROCESSO, 'yyyy/MM/dd'),
      case 
        when length(regexp_replace(DATA_FINALIZACAO_PROCESSO, '[^0-9]', '')) = 6 
        then try_to_date(concat('01', regexp_replace(DATA_FINALIZACAO_PROCESSO, '[^0-9]', '')), 'ddMMyyyy')
      end
    ) AS DATA_FINALIZACAO_PROCESSO,

    coalesce(
      try_to_date(DATA_VENCIMENTO_REGISTRO, 'dd/MM/yyyy'),
      try_to_date(DATA_VENCIMENTO_REGISTRO, 'yyyy-MM-dd'),
      try_to_date(DATA_VENCIMENTO_REGISTRO, 'yyyy/MM/dd'),
      case 
        when length(regexp_replace(DATA_VENCIMENTO_REGISTRO, '[^0-9]', '')) = 6 
        then try_to_date(concat('01', regexp_replace(DATA_VENCIMENTO_REGISTRO, '[^0-9]', '')), 'ddMMyyyy')
      end
    ) AS DATA_VENCIMENTO_REGISTRO,
/*
    coalesce(
      try_to_date(_data_ingestao, 'dd/MM/yyyy'),
      try_to_date(_data_ingestao, 'yyyy-MM-dd'),
      try_to_date(_data_ingestao, 'yyyy/MM/dd')
    ) AS _data_ingestao,
*/
    current_timestamp as _data_ingestao,
    -- 4. Metadado operacional
    "anvisa.b_dados_abertos_medicamentos" AS _arquivo_origem

  FROM anvisa.b_dados_abertos_medicamentos
)

SELECT *
FROM transformed_data
-- 5. Deduplicação por registro e princípio ativo (mantém o primeiro registro encontrado)
QUALIFY row_number() OVER (
  PARTITION BY NUMERO_REGISTRO_PRODUTO, PRINCIPIO_ATIVO 
  ORDER BY DATA_FINALIZACAO_PROCESSO DESC NULLS LAST
) = 1;

In [0]:
%sql
CREATE OR REPLACE TABLE anvisa.s_vigimed_notificacoes AS

WITH treated_data AS (
  SELECT
    -- 1. Chave Primária
    trim(IDENTIFICACAO_NOTIFICACAO) AS IDENTIFICACAO_NOTIFICACAO,

    -- 2. Colunas de Data tratadas para MM/dd/yyyy com timestamp
    coalesce(
      try_to_date(left(trim(DATA_INCLUSAO_SISTEMA), 10), 'MM/dd/yyyy'),
      try_to_date(left(trim(DATA_INCLUSAO_SISTEMA), 10), 'dd/MM/yyyy'),
      try_to_date(left(trim(DATA_INCLUSAO_SISTEMA), 10), 'yyyy-MM-dd')
    ) AS DATA_INCLUSAO_SISTEMA,

    coalesce(
      try_to_date(left(trim(DATA_ULTIMA_ATUALIZACAO), 10), 'MM/dd/yyyy'),
      try_to_date(left(trim(DATA_ULTIMA_ATUALIZACAO), 10), 'dd/MM/yyyy'),
      try_to_date(left(trim(DATA_ULTIMA_ATUALIZACAO), 10), 'yyyy-MM-dd')
    ) AS DATA_ULTIMA_ATUALIZACAO,

    coalesce(
      try_to_date(left(trim(DATA_NOTIFICACAO), 10), 'MM/dd/yyyy'),
      try_to_date(left(trim(DATA_NOTIFICACAO), 10), 'dd/MM/yyyy'),
      try_to_date(left(trim(DATA_NOTIFICACAO), 10), 'yyyy-MM-dd')
    ) AS DATA_NOTIFICACAO,

    coalesce(
      try_to_date(left(trim(DATA_NASCIMENTO), 10), 'MM/dd/yyyy'),
      try_to_date(left(trim(DATA_NASCIMENTO), 10), 'dd/MM/yyyy'),
      try_to_date(left(trim(DATA_NASCIMENTO), 10), 'yyyy-MM-dd')
    ) AS DATA_NASCIMENTO,

    -- 3. Métricas Numéricas Contínuas
    try_cast(replace(trim(IDADE_MOMENTO_REACAO), ',', '.') AS DOUBLE) AS IDADE_MOMENTO_REACAO,
    try_cast(replace(trim(PESO_KG), ',', '.') AS DOUBLE)               AS PESO_KG,
    try_cast(replace(trim(ALTURA_CM), ',', '.') AS DOUBLE)             AS ALTURA_CM,

    -- 4. Variáveis Categóricas e Clínicas Sem Acentuação
    upper(trim(UF)) AS UF,
    upper(trim(SEXO)) AS SEXO,

    translate(upper(trim(TIPO_ENTRADA_VIGIMED)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN') AS TIPO_ENTRADA_VIGIMED,
    translate(upper(trim(RECEBIDO_DE)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN')          AS RECEBIDO_DE,
    translate(upper(trim(TIPO_NOTIFICACAO)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN')     AS TIPO_NOTIFICACAO,
    translate(upper(trim(NOTIFICACAO_PARENT_CHILD)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN') AS NOTIFICACAO_PARENT_CHILD,
    translate(upper(trim(GRUPO_IDADE)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN')          AS GRUPO_IDADE,
    translate(upper(trim(IDADE_GESTACIONAL_MOMENTO_REACAO)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN') AS IDADE_GESTACIONAL_MOMENTO_REACAO,
    translate(upper(trim(REACAO_EVENTO_ADVERSO_MEDDRA)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN') AS REACAO_EVENTO_ADVERSO_MEDDRA,

    -- 5. Flags Binárias
    case 
      when upper(trim(GESTANTE)) in ('SIM', 'S', 'TRUE', '1') then 1
      when upper(trim(GESTANTE)) in ('NAO', 'NÃO', 'N', 'FALSE', '0') then 0
      else null 
    end AS FLG_GESTANTE,

    case 
      when upper(trim(LACTANTE)) in ('SIM', 'S', 'TRUE', '1') then 1
      when upper(trim(LACTANTE)) in ('NAO', 'NÃO', 'N', 'FALSE', '0') then 0
      else null 
    end AS FLG_LACTANTE,

    case 
      when upper(trim(GRAVE)) in ('SIM', 'S', 'TRUE', '1') then 1
      when upper(trim(GRAVE)) in ('NAO', 'NÃO', 'N', 'FALSE', '0') then 0
      else null 
    end AS FLG_GRAVE

  FROM anvisa.b_vigimed_notificacoes
  WHERE IDENTIFICACAO_NOTIFICACAO IS NOT NULL
)

SELECT *
FROM treated_data
QUALIFY row_number() OVER (
  PARTITION BY IDENTIFICACAO_NOTIFICACAO 
  ORDER BY 
    DATA_ULTIMA_ATUALIZACAO DESC NULLS LAST,
    DATA_INCLUSAO_SISTEMA DESC NULLS LAST
) = 1;

In [0]:
%sql
CREATE OR REPLACE TABLE anvisa.s_vigimed_medicamentos AS

WITH treated_data AS (
  SELECT
    -- 1. Chave Estrangeira de Ligação
    trim(IDENTIFICACAO_NOTIFICACAO) AS IDENTIFICACAO_NOTIFICACAO,

    -- 2. Identificação do Fármaco e Princípio Ativo (Sem Acentos)
    translate(upper(trim(PRINCIPIOS_ATIVOS_WHODRUG)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN') AS PRINCIPIOS_ATIVOS_WHODRUG,
    translate(upper(trim(NOME_MEDICAMENTO_WHODRUG)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN')   AS NOME_MEDICAMENTO_WHODRUG,
    upper(trim(CODIGO_ATC)) AS CODIGO_ATC,

    -- 3. Causalidade e Dados Regulamentares
    translate(upper(trim(RELACAO_MEDICAMENTO_EVENTO)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN') AS RELACAO_MEDICAMENTO_EVENTO,
    translate(upper(trim(DETENTOR_REGISTRO)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN')          AS DETENTOR_REGISTRO,
    translate(upper(trim(COMPONENTE_SUSPEITO)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN')        AS COMPONENTE_SUSPEITO,
    translate(upper(trim(ACAO_ADOTADA)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN')               AS ACAO_ADOTADA,
    translate(upper(trim(PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN') AS PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO,

    -- 4. Indicação e Administração Clínica
    translate(upper(trim(INDICACAO_MEDDRA)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN')                     AS INDICACAO_MEDDRA,
    translate(upper(trim(INDICACAO_RELATADA_NOTIFICADOR_INICIAL)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN') AS INDICACAO_RELATADA_NOTIFICADOR_INICIAL,
    translate(upper(trim(FORMA_FARMACEUTICA)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN')                   AS FORMA_FARMACEUTICA,
    translate(upper(trim(VIA_ADMINISTRACAO)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN')                    AS VIA_ADMINISTRACAO,

    -- 5. Posologia e Dosagem
    upper(trim(CONCENTRACAO))    AS CONCENTRACAO,
    upper(trim(DOSE))            AS DOSE,
    upper(trim(FREQUENCIA_DOSE)) AS FREQUENCIA_DOSE,
    upper(trim(POSOLOGIA))       AS POSOLOGIA,
    upper(trim(DURACAO))         AS DURACAO,

    -- 6. Tratamento Robusto de Datas (Suporte a yyyyMMdd e formatos com delimitadores)
    case 
      when length(regexp_replace(trim(INICIO_ADMINISTRACAO), '[^0-9]', '')) = 8 
      then try_to_date(regexp_replace(trim(INICIO_ADMINISTRACAO), '[^0-9]', ''), 'yyyyMMdd')
      else coalesce(
        try_to_date(left(trim(INICIO_ADMINISTRACAO), 10), 'yyyy-MM-dd'),
        try_to_date(left(trim(INICIO_ADMINISTRACAO), 10), 'dd/MM/yyyy'),
        try_to_date(left(trim(INICIO_ADMINISTRACAO), 10), 'MM/dd/yyyy')
      )
    end AS INICIO_ADMINISTRACAO,

    case 
      when length(regexp_replace(trim(FIM_ADMINISTRACAO), '[^0-9]', '')) = 8 
      then try_to_date(regexp_replace(trim(FIM_ADMINISTRACAO), '[^0-9]', ''), 'yyyyMMdd')
      else coalesce(
        try_to_date(left(trim(FIM_ADMINISTRACAO), 10), 'yyyy-MM-dd'),
        try_to_date(left(trim(FIM_ADMINISTRACAO), 10), 'dd/MM/yyyy'),
        try_to_date(left(trim(FIM_ADMINISTRACAO), 10), 'MM/dd/yyyy')
      )
    end AS FIM_ADMINISTRACAO

  FROM anvisa.b_vigimed_medicamentos
  WHERE IDENTIFICACAO_NOTIFICACAO IS NOT NULL
)

SELECT *
FROM treated_data
-- 7. Deduplicação: Mantém apenas um registo por notificação e medicamento/princípio ativo
QUALIFY row_number() OVER (
  PARTITION BY IDENTIFICACAO_NOTIFICACAO, PRINCIPIOS_ATIVOS_WHODRUG, NOME_MEDICAMENTO_WHODRUG
  ORDER BY INICIO_ADMINISTRACAO DESC NULLS LAST
) = 1;

In [0]:
%sql
CREATE OR REPLACE TABLE anvisa.s_vigimed_reacoes AS

WITH treated_data AS (
  SELECT
    -- 1. Chave Estrangeira de Ligação
    trim(IDENTIFICACAO_NOTIFICACAO) AS IDENTIFICACAO_NOTIFICACAO,

    -- 2. Hierarquia MedDRA Sem Acentos
    translate(upper(trim(REACAO_EVTO_ADVERSO_MEDDRA_LLT)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN') AS REACAO_EVTO_ADVERSO_MEDDRA_LLT,
    translate(upper(trim(PT)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN')                             AS PT,
    translate(upper(trim(HLT)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN')                            AS HLT,
    translate(upper(trim(HLGT)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN')                           AS HLGT,
    translate(upper(trim(SOC)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN')                            AS SOC,

    -- 3. Tratamento Flexível de Datas
    case 
      when length(regexp_replace(trim(DATA_INICIO_HORA), '[^0-9]', '')) = 8 
      then try_to_date(regexp_replace(trim(DATA_INICIO_HORA), '[^0-9]', ''), 'yyyyMMdd')
      else coalesce(
        try_to_date(left(trim(DATA_INICIO_HORA), 10), 'MM/dd/yyyy'),
        try_to_date(left(trim(DATA_INICIO_HORA), 10), 'yyyy-MM-dd'),
        try_to_date(left(trim(DATA_INICIO_HORA), 10), 'dd/MM/yyyy')
      )
    end AS DATA_INICIO_HORA,

    case 
      when length(regexp_replace(trim(DATA_FINAL_HORA), '[^0-9]', '')) = 8 
      then try_to_date(regexp_replace(trim(DATA_FINAL_HORA), '[^0-9]', ''), 'yyyyMMdd')
      else coalesce(
        try_to_date(left(trim(DATA_FINAL_HORA), 10), 'MM/dd/yyyy'),
        try_to_date(left(trim(DATA_FINAL_HORA), 10), 'yyyy-MM-dd'),
        try_to_date(left(trim(DATA_FINAL_HORA), 10), 'dd/MM/yyyy')
      )
    end AS DATA_FINAL_HORA,

    -- 4. Duração e Indicadores Clínicos
    upper(trim(DURACAO)) AS DURACAO,

    case 
      when upper(trim(GRAVE)) in ('SIM', 'S', 'TRUE', '1') then 1
      when upper(trim(GRAVE)) in ('NAO', 'NÃO', 'N', 'FALSE', '0') then 0
      else null 
    end AS FLG_GRAVE,

    translate(upper(trim(GRAVIDADE)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN') AS GRAVIDADE,
    translate(upper(trim(DESFECHO)), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ', 'AAAAAEEEEIIIIOOOOOUUUUCN')  AS DESFECHO,

    -- 5. Metadados Operacionais
    current_timestamp() AS _data_ingestao,
    "anvisa.b_vigimed_reacoes" AS _arquivo_origem

  FROM anvisa.b_vigimed_reacoes
  WHERE IDENTIFICACAO_NOTIFICACAO IS NOT NULL
)

SELECT *
FROM treated_data
-- 6. Deduplicação: Garante que uma dada reação/sintoma (PT) não se repita no mesmo caso
QUALIFY row_number() OVER (
  PARTITION BY IDENTIFICACAO_NOTIFICACAO, PT
  ORDER BY DATA_INICIO_HORA DESC NULLS LAST
) = 1;

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, Imputer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
import pandas as pd

# 1. Carregamento da tabela Gold
df_gold = spark.read.table("anvisa.g_vigimed_features_ml")

# Garantir que o target seja do tipo numérico Double (exigência do PySpark MLlib)
df_ml = df_gold.withColumn("TARGET_IS_GRAVE", F.col("TARGET_IS_GRAVE").cast("double"))

# 2. Definição dos grupos de colunas
categorical_cols = [
    "SEXO", 
    "UF", 
    "TIPO_NOTIFICACAO", 
    "RECEBIDO_DE"
]

continuous_cols = [
    "IDADE_ANOS", 
    "PESO_KG", 
    "ALTURA_CM", 
    "IMC", 
    "DIAS_ATE_INCLUSAO_SISTEMA"
]

binary_and_count_cols = [
    "FLG_GESTANTE", "FLG_LACTANTE",
    "QTD_PRINCIPIOS_ATIVOS", "QTD_MEDICAMENTOS_TOTAL",
    "QTD_FARMACOS_SUSPEITOS", "QTD_FARMACOS_CONCOMITANTES", "QTD_FARMACOS_INTERACAO",
    "FLG_VIA_INTRAVENOSA", "FLG_VIA_INTRAMUSCULAR", "FLG_VIA_SUBCUTANEA",
    "FLG_VIA_ENTERAL_SONDA", "FLG_VIA_RESPIRATORIA", "FLG_VIA_NEUROAXIAL_CAVITARIA",
    "FLG_VIA_OFTALMICA_OTOLOGICA", "FLG_VIA_TOPICA_CUTANEA", "FLG_VIA_RETAL_UROGENITAL",
    "FLG_VIA_ORAL", "FLG_VIA_PARENTERAL_INESPECIFICA", "FLG_VIA_MATERNO_FETAL",
    "FLG_VIA_NAO_INFORMADA",
    "FLG_USA_GENERICO", "FLG_USA_SIMILAR", "FLG_USA_NOVO", "FLG_USA_BIOLOGICO",
    "QTD_REACOES_DISTINTAS", "QTD_SISTEMAS_AFETADOS",
    "FLG_SOC_CARDIACO", "FLG_SOC_GASTROINTESTINAL", "FLG_SOC_RESPIRATORIO",
    "FLG_SOC_NERVOSO", "FLG_SOC_CUTANEO"
]

# Garantir que as colunas contínuas estão como double para o Imputer
for c in continuous_cols:
    df_ml = df_ml.withColumn(c, F.col(c).cast("double"))

# Substituir eventuais nulos nas categóricas por uma string padrão 'NAO_INFORMADO'
df_ml = df_ml.fillna("NAO_INFORMADO", subset=categorical_cols)
df_ml = df_ml.fillna(0, subset=binary_and_count_cols)

# 3. Divisão em Treino (80%) e Teste (20%)
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

print(f"Registros de Treino: {train_df.count()}")
print(f"Registros de Teste:  {test_df.count()}")

# 4. Construção das Etapas do Pipeline de Pré-processamento
# 4.1. Indexação e One-Hot Encoding para variáveis categóricas (Texto -> Número -> Vetor Binário)
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") 
    for c in categorical_cols
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in categorical_cols],
    outputCols=[f"{c}_ohe" for c in categorical_cols],
    handleInvalid="keep"
)

# 4.2. Imputação pela mediana para variáveis contínuas (trata nulos em Peso, Altura, Idade, IMC)
imputed_cols = [f"{c}_imp" for c in continuous_cols]
imputer = Imputer(
    inputCols=continuous_cols,
    outputCols=imputed_cols,
    strategy="median"
)

# 4.3. VectorAssembler: Une todas as features em um único vetor 'features'
assembler_inputs = [f"{c}_ohe" for c in categorical_cols] + imputed_cols + binary_and_count_cols
assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features",
    handleInvalid="keep"
)

# 5. Definição do Modelo Random Forest Básico
rf = RandomForestClassifier(
    labelCol="TARGET_IS_GRAVE",
    featuresCol="features",
    numTrees=100,      # Quantidade de árvores na floresta
    maxDepth=8,        # Profundidade máxima de cada árvore (evita overfitting)
    seed=42
)

# 6. Montagem e Treinamento do Pipeline
pipeline = Pipeline(stages=indexers + [encoder, imputer, assembler, rf])
modelo_rf = pipeline.fit(train_df)

# 7. Geração das Previsões na Base de Teste
predictions = modelo_rf.transform(test_df)

# 8. Avaliação de Acurácia e Métricas Complementares
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="TARGET_IS_GRAVE", predictionCol="prediction", metricName="accuracy"
)
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="TARGET_IS_GRAVE", predictionCol="prediction", metricName="f1"
)
evaluator_prec = MulticlassClassificationEvaluator(
    labelCol="TARGET_IS_GRAVE", predictionCol="prediction", metricName="weightedPrecision"
)
evaluator_rec = MulticlassClassificationEvaluator(
    labelCol="TARGET_IS_GRAVE", predictionCol="prediction", metricName="weightedRecall"
)
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="TARGET_IS_GRAVE", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)

acuracia = evaluator_acc.evaluate(predictions)
f1_score = evaluator_f1.evaluate(predictions)
precisao = evaluator_prec.evaluate(predictions)
recall   = evaluator_rec.evaluate(predictions)
auc_roc  = evaluator_auc.evaluate(predictions)

print("=" * 45)
print("RESULTADOS DE AVALIAÇÃO DO RANDOM FOREST")
print("=" * 45)
print(f"Acurácia (Accuracy): {acuracia:.4f} ({acuracia*100:.2f}%)")
print(f"F1-Score:            {f1_score:.4f}")
print(f"Precisão Ponderada:  {precisao:.4f}")
print(f"Recall Ponderado:    {recall:.4f}")
print(f"AUC-ROC:             {auc_roc:.4f}")
print("=" * 45)

# 9. Matriz de Confusão (Real vs. Previsto)
print("\nMatriz de Confusão (0 = Não Grave, 1 = Grave):")
predictions.groupBy("TARGET_IS_GRAVE", "prediction").count().orderBy("TARGET_IS_GRAVE", "prediction").show()